mouse-heatmap

In [ ]:
# ============================================================
# - 读取 mlp_tanh benchmark summary
# - 按 Threshold 分别绘制 Precision / Pearson_R / MAE / RMSE 热图
# ============================================================

import os
import platform
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.font_manager as font_manager

# ==========================================
# 1. 顶级期刊全局配置
# ==========================================
system = platform.system()
if system == "Windows":
    font_path = "C:/Windows/Fonts/arial.ttf"
elif system == "Darwin":
    font_path = "/Library/Fonts/Arial.ttf"
else:
    font_path = "/mnt/c/Windows/Fonts/arial.ttf"
    if not os.path.exists(font_path):
        font_path = "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf"

if os.path.exists(font_path):
    font_manager.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = "Arial"
else:
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["axes.linewidth"] = 0.5

# ==========================================
# 2. 输入输出路径
# ==========================================
summary_csv_path = "/mnt/e/1-TMS-remove/4-precision-5_to_25/1-5_to_25-pcc-add-model-mlp-tanh-parallel/1-end-mlp_tanh_random_sample_summary_parallel.csv"
# 如果你运行的是非并行版本，可改为：
# summary_csv_path = "/mnt/e/1-TMS-remove/4-precision-5_to_25/1-5_to_25-pcc-add-model-mlp-tanh/1-end-mlp_tanh_random_sample_summary.csv"

save_dir = "/mnt/e/2-8.3-shanda/1-feature/1-figure/0-2-result-3-benchmark/1-heatmap-mouse-mlp-tanh"
os.makedirs(save_dir, exist_ok=True)

models_to_include = {"scimmuaging", "buckley", "scale", "maple", "xgboost", "iage"}

# ==========================================
# 3. 色板和指标配置
# ==========================================
colors_bwr = ["#1D4E89", "#73A5C6", "#FDFDFD", "#F26D5B", "#9E1A1A"]
cmap_bwr = mcolors.LinearSegmentedColormap.from_list("BlueWhiteRed", colors_bwr, N=256)
cmap_rwb = cmap_bwr.reversed(name="RedWhiteBlue")

metrics_config = {
    "Precision": {
        "cmap": cmap_bwr,
        "sort_ascending": False,
        "short_label": "Precision",
        "fmt": ".2f",
        "center_percentile": 60,
    },
    "Pearson_R": {
        "cmap": cmap_bwr,
        "sort_ascending": False,
        "short_label": "Pearson R",
        "fmt": ".2f",
        "center_percentile": 60,
    },
    "MAE": {
        "cmap": cmap_rwb,
        "sort_ascending": True,
        "short_label": "MAE(months)",
        "fmt": ".2f",
        "center_percentile": 40,
    },
    "RMSE": {
        "cmap": cmap_bwr,
        "sort_ascending": True,
        "short_label": "RMSE",
        "fmt": ".2f",
        "center_percentile": 40,
    },
}

MODEL_DISPLAY_NAME = {
    "maple": "Sage",
}


def clean_model_label(model):
    model_str = str(model).strip()
    model_lower = model_str.lower()
    if model_lower == "maple":
        return "Sage"
    if model_lower == "scimmuaging":
        return "sc-ImmuAging"
    if model_lower == "iage":
        return "iAge"
    if model_lower == "xgboost":
        return "XGboost"
    return model_str[:1].upper() + model_str[1:].lower()


def clean_tissue_label(tissue):
    return str(tissue).replace("_", " ")


def safe_name(value):
    return str(value).replace("/", "-").replace(" ", "_").replace(".", "_")


def plot_one_heatmap(df_filtered, metric, config, threshold, benchmark_name=None, color_limits=None):
    value_df = df_filtered.dropna(subset=[metric]).copy()
    if value_df.empty:
        return None

    pivot_df = value_df.pivot_table(index="Tissue", columns="Model", values=metric, aggfunc="mean")
    pivot_df = pivot_df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if pivot_df.empty:
        return None

    model_mean_scores = pivot_df.mean(axis=0).sort_values(ascending=config["sort_ascending"])
    pivot_df = pivot_df[model_mean_scores.index]

    # 与原图保持一致：如果有 maple/Sage，把 Sage 放在第一列；组织按 maple 排序。
    if "maple" in pivot_df.columns:
        pivot_df = pivot_df.sort_values(by="maple", ascending=config["sort_ascending"])
        cols = list(pivot_df.columns)
        cols.insert(0, cols.pop(cols.index("maple")))
        pivot_df = pivot_df[cols]
    else:
        # 没有 maple 时，用排序后第一列作为组织排序参考。
        first_col = pivot_df.columns[0]
        pivot_df = pivot_df.sort_values(by=first_col, ascending=config["sort_ascending"])

    pivot_df.index = [clean_tissue_label(x) for x in pivot_df.index]
    pivot_df.columns = [clean_model_label(x) for x in pivot_df.columns]

    vals = pivot_df.values.astype(float)
    finite_vals = vals[np.isfinite(vals)]
    if finite_vals.size == 0:
        return None

    if color_limits is not None and metric in color_limits:
        global_min, global_max, plot_center = color_limits[metric]
    else:
        global_min = float(np.nanmin(finite_vals))
        global_max = float(np.nanmax(finite_vals))
        plot_center = float(np.nanpercentile(finite_vals, config["center_percentile"]))

    if not np.isfinite(global_min) or not np.isfinite(global_max) or global_max <= global_min:
        global_min = float(np.nanmin(finite_vals))
        global_max = float(np.nanmax(finite_vals))

    if not np.isfinite(global_min) or not np.isfinite(global_max) or global_max <= global_min:
        global_min, global_max = 0.0, 1.0

    if not np.isfinite(plot_center):
        plot_center = (global_min + global_max) / 2.0

    width_in = 100 / 25.4
    height_in = 80 / 25.4
    fig, ax = plt.subplots(figsize=(width_in, height_in))

    sns.heatmap(
        pivot_df,
        annot=True,
        cmap=config["cmap"],
        vmin=global_min,
        vmax=global_max,
        center=plot_center,
        robust=False,
        fmt=config["fmt"],
        annot_kws={"size": 6, "family": "Arial", "color": "black"},
        cbar_kws={"shrink": 0.6, "aspect": 20, "pad": 0.04},
        ax=ax,
    )

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=6, family="Arial")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=6, family="Arial")
    ax.tick_params(axis="both", which="major", length=2.5, width=0.5, color="black", direction="out", pad=2)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=6, width=0.5, length=2.5, pad=2)
    cbar.outline.set_visible(False)
    cbar.ax.set_title(config["short_label"], size=6, family="Arial", pad=6, loc="left")

    if benchmark_name is not None:
        ax.set_title(str(benchmark_name).replace("_", " "), fontsize=7, pad=4)

    plt.tight_layout()

    if benchmark_name is None:
        save_name = f"MLP_Tanh_Heatmap_{metric}_Threshold{threshold}.pdf"
    else:
        save_name = f"MLP_Tanh_Heatmap_{metric}_Threshold{threshold}_{safe_name(benchmark_name)}.pdf"

    save_path = os.path.join(save_dir, save_name)
    plt.savefig(save_path, format="pdf", bbox_inches="tight", facecolor="white", transparent=False)
    plt.close(fig)
    return save_path

# ==========================================
# 4. 批量生成热图
# ==========================================
if not os.path.exists(summary_csv_path):
    print(f"找不到结果文件，请检查路径: {summary_csv_path}")
else:
    df_summary = pd.read_csv(summary_csv_path)
    df_summary = df_summary[
        df_summary["Model"].astype(str).str.lower().isin(models_to_include)
    ].copy()

    required_cols = {"Model", "Benchmark_Name", "Tissue", "Threshold", "Precision", "Pearson_R", "MAE", "RMSE"}
    missing_cols = required_cols.difference(df_summary.columns)
    if missing_cols:
        raise ValueError(f"summary CSV 缺少必要列: {sorted(missing_cols)}")

    all_thresholds = sorted(df_summary["Threshold"].dropna().unique())

    # 为每个指标预先计算统一的 colorbar 范围。
    # 这样不同基因数量 Threshold 下，同一个指标使用相同的 vmin/vmax/center，颜色可以直接比较。
    metric_color_limits = {}
    for metric, config in metrics_config.items():
        vals = pd.to_numeric(df_summary[metric], errors="coerce").dropna().values.astype(float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            metric_color_limits[metric] = (0.0, 1.0, 0.5)
            continue
        vmin = float(np.nanmin(vals))
        vmax = float(np.nanmax(vals))
        center = float(np.nanpercentile(vals, config["center_percentile"]))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            vmin, vmax = 0.0, 1.0
        if not np.isfinite(center):
            center = (vmin + vmax) / 2.0
        metric_color_limits[metric] = (vmin, vmax, center)
        print(f"{metric} colorbar: vmin={vmin:.4g}, center={center:.4g}, vmax={vmax:.4g}")

    saved_files = []

    # A. 与原 heatmap 代码一致：跨 benchmark 平均后，每个 threshold/metric 一张总图。
    for threshold in all_thresholds:
        print(f"\n正在处理总体热图 Threshold = {threshold}")
        df_threshold = df_summary[df_summary["Threshold"] == threshold]
        if df_threshold.empty:
            continue
        for metric, config in metrics_config.items():
            save_path = plot_one_heatmap(
                df_threshold,
                metric,
                config,
                threshold,
                benchmark_name=None,
                color_limits=metric_color_limits,
            )
            if save_path:
                saved_files.append(save_path)
                print(f"    已保存: {save_path}")

    # B. Precision 受 Benchmark_Name 影响较大，额外按 benchmark 分开保存 Precision 图，方便查看。
    for threshold in all_thresholds:
        df_threshold = df_summary[df_summary["Threshold"] == threshold]
        for benchmark_name in sorted(df_threshold["Benchmark_Name"].dropna().unique()):
            df_bench = df_threshold[df_threshold["Benchmark_Name"] == benchmark_name]
            save_path = plot_one_heatmap(
                df_bench,
                "Precision",
                metrics_config["Precision"],
                threshold,
                benchmark_name=benchmark_name,
                color_limits=metric_color_limits,
            )
            if save_path:
                saved_files.append(save_path)
                print(f"    已保存 benchmark Precision: {save_path}")

    pd.DataFrame({"figure_path": saved_files}).to_csv(
        os.path.join(save_dir, "MLP_Tanh_Heatmap_Figure_Index.csv"),
        index=False,
    )

    print(f"\n完成。共保存 {len(saved_files)} 张 PDF 热图。")
    print(f"输出目录: {save_dir}")
